# Taller 2 de MLOps

 - Andres Felipe Hernandez
 - Andres Aristizabal
 - Juan José Salazar

## 1. Carpetas habituales en Linux

Linux organiza archivos desde `/`. Estas carpetas tienen propósitos convencionales definidos por el estándar de jerarquía del sistema de archivos (FHS).

| Carpeta | Uso principal |
|---|---|
| `/bin` | Comandos esenciales para todos los usuarios, por ejemplo `ls` y `cp`. En distribuciones actuales puede ser un enlace a `/usr/bin`. |
| `/opt` | Aplicaciones adicionales instaladas como paquetes autocontenidos. |
| `/boot` | Archivos necesarios para iniciar el sistema, como el kernel y la configuración del cargador de arranque. |
| `/dev` | Archivos especiales que representan dispositivos, por ejemplo discos y terminales. |
| `/sbin` | Herramientas esenciales de administración del sistema; a menudo enlaza con `/usr/sbin`. |
| `/etc` | Archivos de configuración del sistema y de servicios. |
| `/home` | Directorios personales de los usuarios. |
| `/srv` | Datos que ofrecen servicios del sistema, por ejemplo contenido de un servidor web. |
| `/lib` | Bibliotecas compartidas esenciales y módulos del kernel; puede enlazar con `/usr/lib`. |
| `/tmp` | Archivos temporales; su contenido puede borrarse automáticamente. |
| `/usr` | Programas, bibliotecas y documentación de uso general. **No** significa “user files”; los datos personales suelen ir en `/home`. |
| `/media` | Puntos de montaje para medios removibles, como memorias USB. |
| `/var` | Datos variables, como registros, cachés y colas. |
| `/mnt` | Punto de montaje temporal usado por el administrador. |

La organización exacta puede variar entre distribuciones; por ejemplo, `/bin`, `/sbin` y `/lib` suelen estar integradas con sus equivalentes bajo `/usr`.


## 2. Operadores de Bash

- `>` redirige la salida estándar a un archivo y lo sobrescribe (o lo crea).
- `&&` ejecuta el segundo comando solo si el primero termina correctamente.
- `|` envía la salida de un comando como entrada del siguiente.

Son operadores de shell; no son los operadores de comparación usados en `if`.


In [ ]:
# > crea o sobrescribe el archivo
printf 'Hola desde Bash\n' > saludo.txt
cat saludo.txt

# && ejecuta el segundo comando si el primero tuvo éxito
mkdir -p ejemplo && cd ejemplo
pwd

# | conecta la salida de ls con grep para filtrar nombres
ls /usr/bin | grep '^python' | head -n 5


## 3. Comodines (wildcards)

Los comodines permiten seleccionar nombres de archivos según patrones. El shell expande el patrón antes de ejecutar el comando. `*` representa cero o más caracteres; `?` representa exactamente un carácter; `[abc]` representa uno de los caracteres listados y `[0-9]` un carácter dentro del rango.


In [ ]:
# Ejemplos de patrones (la expansión depende de los archivos existentes)
ls *.txt       # todos los nombres terminados en .txt
ls informe?.csv # informe1.csv, informeA.csv, etc.
ls foto[12].jpg # foto1.jpg o foto2.jpg


## 4. Comandos `find` y `grep`

`find` busca archivos y directorios recorriendo una ruta, con filtros como nombre o tipo. `grep` busca líneas que coinciden con un texto o una expresión regular. Se pueden combinar con una tubería.


In [ ]:
# Buscar archivos .py bajo el directorio actual
find . -type f -name '*.py'

# Buscar una palabra ignorando mayúsculas en archivos .txt
grep -in 'error' ./*.txt

# Buscar una cadena en los archivos .log encontrados bajo logs/
find logs -type f -name '*.log' -print0 | xargs -0 grep -n 'timeout'


## 5. Revisar y finalizar procesos desde la terminal

Sí. `ps` muestra procesos, `top` ofrece una vista que se actualiza y `pgrep` permite localizar procesos por nombre. Para finalizar uno, se envía una señal con `kill PID`; primero se recomienda la señal normal `TERM` y, si no responde, `KILL` como último recurso. Se requieren permisos adecuados.


In [ ]:
# Consultar procesos y buscar uno por nombre
ps aux | head
pgrep -a python

# Finalización ordenada (sustituir PID por el identificador real)
kill PID

# Solo si no termina con TERM: forzar la finalización
kill -9 PID


## 6. Ramas en Git y Pull Request

Una rama es una referencia móvil a una línea de trabajo basada en commits. Permite desarrollar una función o corrección sin modificar directamente la rama principal y facilita revisar y combinar cambios. El flujo solicitado por el taller requiere hacer el Pull Request en el repositorio de un compañero.

### Ejemplo de flujo



In [ ]:
# Clonar el repositorio del compañero y crear una rama
git clone https://github.com/JUANJOSEDH028/repoTaller2
cd nombre-del-repositorio
git switch -c docs/punto-6

# Después de editar/agregar los archivos solicitados
git status
git add archivo-modificado
git commit -m "Documenta el punto 6 del taller"
git push -u origin docs/punto-6


## 7. Volver a un estado anterior en Git

Un commit identifica un estado del proyecto. Para inspeccionar uno anterior sin mover la rama se puede usar `git switch --detach <commit>`. Para deshacer cambios en una rama compartida, `git revert <commit>` crea un commit nuevo que revierte el anterior. `git reset` mueve la rama y puede descartar trabajo, por lo que se usa con cuidado, especialmente si los commits ya se compartieron.


In [ ]:
# Ver historial resumido y copiar el hash del commit deseado
git log --oneline

# Inspeccionar temporalmente el proyecto en ese commit
git switch --detach HASH_DEL_COMMIT

# Volver a la rama de trabajo
git switch main

# Alternativa segura para deshacer un commit ya compartido: crea un commit inverso
git revert HASH_DEL_COMMIT


### ¿Cuál alternativa elegir?

- **Solo quiero revisar o ejecutar una versión antigua:** `git switch --detach HASH`. Luego regreso a mi rama con `git switch main` (o el nombre de mi rama).
- **Quiero deshacer un cambio en una rama compartida:** `git revert HASH`; conserva el historial y publica la reversión como un commit nuevo.
- **Quiero mover una rama local a un commit anterior:** `git reset --hard HASH` cambia el historial local y elimina cambios no guardados. No se recomienda para una rama compartida.

En un equipo, `revert` suele ser la opción más segura para corregir un commit que ya se publicó.


## 8. Optuna: persistencia de estudios en SQLite

Un `study` de Optuna puede guardarse en una base de datos relacional mediante `storage`. SQLite es útil para este ejemplo porque guarda todo en un archivo local y permite continuar el estudio después de reiniciar el proceso.

### Tablas internas más relevantes

| Tabla | Información almacenada |
|---|---|
| `studies` | Identidad y nombre de cada estudio. |
| `study_directions` | Dirección objetivo de cada estudio: minimizar o maximizar. |
| `study_user_attributes` / `study_system_attributes` | Atributos asociados al estudio, definidos por el usuario o por Optuna. |
| `trials` | Cada intento de optimización, su número, estado y vínculo con el estudio. |
| `trial_values` | Valores objetivo finales de los trials. |
| `trial_params` | Parámetros sugeridos y sus valores para cada trial. |
| `trial_user_attributes` / `trial_system_attributes` | Metadatos asociados a los trials. |
| `trial_intermediate_values` | Métricas intermedias, útiles para pruning. |
| `trial_heartbeats` | Señales de vida de trials en ejecución, cuando se habilita el mecanismo correspondiente. |
| `version_info` | Versión del esquema de almacenamiento de Optuna. |

El esquema preciso puede variar entre versiones de Optuna. El ejemplo siguiente consulta las tablas creadas por la versión instalada.

El notebook original de Optuna se incorpora a continuación. En Google Colab, monta Google Drive si deseas conservar allí el archivo SQLite entre sesiones; localmente puedes cambiar `DB_PATH` por una ruta local.


In [1]:
!pip install optuna -q

In [1]:
import os
import sqlite3
import optuna
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)
print(optuna.__version__)

5.0.0


In [2]:
X, y = load_breast_cancer(return_X_y=True)
print(X.shape, y.shape)
print("Clases:", load_breast_cancer().target_names)

(569, 30) (569,)
Clases: ['malignant' 'benign']


In [3]:
def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 300)
    max_depth = trial.suggest_int("max_depth", 2, 20, log=True)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    criterion = trial.suggest_categorical("criterion", ["gini", "entropy"])

    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        criterion=criterion,
        random_state=42,
        n_jobs=-1,
    )
    score = cross_val_score(clf, X, y, cv=5, scoring="roc_auc").mean()

    trial.set_user_attr("modelo", "RandomForest")
    return score

In [4]:
# Guardar el estudio en una base SQLite local, junto al notebook
DB_PATH = "optuna_cancer.db"
STORAGE = f"sqlite:///{DB_PATH}"
STUDY_NAME = "rf_breast_cancer"

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE,
    direction="maximize",
    load_if_exists=True,
)
study.set_user_attr("dataset", "breast_cancer")
print("Study creado/cargado:", study.study_name)


Study creado/cargado: rf_breast_cancer


In [5]:
study.optimize(objective, n_trials=20)

print("Mejor AUC:", round(study.best_value, 4))
print("Mejores parámetros:", study.best_params)
print("Trials en la BD:", len(study.trials))

Mejor AUC: 0.9933
Mejores parámetros: {'n_estimators': 61, 'max_depth': 19, 'min_samples_leaf': 6, 'criterion': 'entropy'}
Trials en la BD: 20


In [6]:
del study
study = optuna.load_study(study_name=STUDY_NAME, storage=STORAGE)
print("Trials recuperados:", len(study.trials))

study.optimize(objective, n_trials=20)
print("Trials totales:", len(study.trials))
print("Mejor AUC:", round(study.best_value, 4), study.best_params)

Trials recuperados: 20
Trials totales: 40
Mejor AUC: 0.9933 {'n_estimators': 100, 'max_depth': 10, 'min_samples_leaf': 5, 'criterion': 'entropy'}


In [7]:
con = sqlite3.connect(DB_PATH)
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", con)

,name
0,alembic_version
1,studies
2,study_directions
3,study_system_attributes
4,study_user_attributes
5,trial_heartbeats
6,trial_intermediate_values
7,trial_params
8,trial_system_attributes
9,trial_user_attributes


In [8]:
display(pd.read_sql("SELECT * FROM studies", con))
display(pd.read_sql("SELECT * FROM study_directions", con))

,study_id,study_name
0,1,rf_breast_cancer


,study_direction_id,direction,study_id,objective
0,1,MAXIMIZE,1,0


In [9]:
pd.read_sql("SELECT * FROM trials LIMIT 5", con)

,trial_id,number,study_id,state,datetime_start,datetime_complete
0,1,0,1,COMPLETE,2026-11-09 02:33:33.051223,2026-11-09 02:33:35.198901
1,2,1,1,COMPLETE,2026-11-09 02:33:35.219346,2026-11-09 02:33:37.644017
2,3,2,1,COMPLETE,2026-11-09 02:33:37.667639,2026-11-09 02:33:40.300940
3,4,3,1,COMPLETE,2026-11-09 02:33:40.319607,2026-11-09 02:33:41.092820
4,5,4,1,COMPLETE,2026-11-09 02:33:41.115358,2026-11-09 02:33:44.070710


In [10]:
pd.set_option("display.max_colwidth", None)
pd.read_sql("SELECT * FROM trial_params WHERE trial_id = 1", con)

,param_id,trial_id,param_name,param_value,distribution_json
0,4,1,criterion,1.0,"{""name"": ""CategoricalDistribution"", ""attributes"": {""choices"": [""gini"", ""entropy""]}}"
1,2,1,max_depth,7.0,"{""name"": ""IntDistribution"", ""attributes"": {""log"": true, ""step"": 1, ""low"": 2, ""high"": 20}}"
2,3,1,min_samples_leaf,8.0,"{""name"": ""IntDistribution"", ""attributes"": {""log"": false, ""step"": 1, ""low"": 1, ""high"": 10}}"
3,1,1,n_estimators,183.0,"{""name"": ""IntDistribution"", ""attributes"": {""log"": false, ""step"": 1, ""low"": 10, ""high"": 300}}"


In [11]:
pd.read_sql("SELECT * FROM trial_values ORDER BY value DESC LIMIT 5", con)

,trial_value_id,trial_id,objective,value,value_type
0,36,36,0,0.993292,FINITE
1,19,19,0,0.993290,FINITE
2,37,37,0,0.993159,FINITE
3,40,40,0,0.993096,FINITE
4,38,38,0,0.993028,FINITE


In [12]:
display(pd.read_sql("SELECT * FROM study_user_attributes", con))
display(pd.read_sql("SELECT * FROM trial_user_attributes LIMIT 3", con))
display(pd.read_sql("SELECT * FROM version_info", con))

,study_user_attribute_id,study_id,key,value_json
0,1,1,dataset,"""breast_cancer"""


,trial_user_attribute_id,trial_id,key,value_json
0,1,1,modelo,"""RandomForest"""
1,2,2,modelo,"""RandomForest"""
2,3,3,modelo,"""RandomForest"""


,version_info_id,schema_version,library_version
0,1,12,5.0.0


In [13]:
query = """
SELECT t.number, t.state, v.value AS auc,
       MAX(CASE WHEN p.param_name='n_estimators'     THEN p.param_value END) AS n_estimators,
       MAX(CASE WHEN p.param_name='max_depth'        THEN p.param_value END) AS max_depth,
       MAX(CASE WHEN p.param_name='min_samples_leaf' THEN p.param_value END) AS min_samples_leaf,
       MAX(CASE WHEN p.param_name='criterion'        THEN p.param_value END) AS criterion
FROM trials t
JOIN trial_values v ON v.trial_id = t.trial_id
JOIN trial_params p ON p.trial_id = t.trial_id
GROUP BY t.trial_id
ORDER BY auc DESC
LIMIT 5
"""
pd.read_sql(query, con)

,number,state,auc,n_estimators,max_depth,min_samples_leaf,criterion
0,35,COMPLETE,0.993292,100.0,10.0,5.0,1.0
1,18,COMPLETE,0.993290,61.0,19.0,6.0,1.0
2,36,COMPLETE,0.993159,93.0,12.0,5.0,1.0
3,39,COMPLETE,0.993096,115.0,11.0,5.0,1.0
4,37,COMPLETE,0.993028,95.0,8.0,6.0,1.0


In [14]:
con.close()

## 9. MLflow Model Registry: versiones y alias

### Conceptos

**Guardar un modelo en una ejecución** con `mlflow.sklearn.log_model` almacena el modelo como artefacto de una ejecución junto con sus parámetros y métricas. Se referencia mediante una URI como `runs:/<run_id>/model`; por sí solo no ofrece el historial ni los alias del registro.

**Registrar el modelo** con `mlflow.register_model` lo incorpora al Model Registry bajo un nombre estable (`iris_classifier`). Cada registro recibe una versión y mantiene el vínculo con la ejecución de origen. Las ejecuciones sirven para experimentar y el registro para gestionar los modelos disponibles.

Una **versión** es un número asignado por MLflow y apunta a artefactos concretos. Un **alias** como `champion` o `candidate` es una etiqueta que se puede reasignar a distintas versiones. La versión más reciente no necesariamente es la mejor: el número solo refleja el orden de registro, así que la promoción debe decidirse comparando métricas.

### Implementación

El ejemplo usa Iris, una división estratificada 70/30 y semilla 42. Entrena dos Random Forest de 50 árboles (`max_depth=1` y `max_depth=4`) en ejecuciones independientes, los registra bajo el mismo nombre, compara accuracy, carga por alias y simula un rollback.

Instala las dependencias en un entorno que permita ejecutar Jupyter. La base `mlflow.db` y los artefactos se crearán en el directorio de trabajo.


In [ ]:
%pip install -q "mlflow>=2.19,<4" scikit-learn


In [ ]:
import inspect

import mlflow
import mlflow.pyfunc
import mlflow.sklearn
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# --- Configuración -----------------------------------------------------------
# El Model Registry necesita un backend con base de datos; SQLite basta en local.
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("iris_registry_demo")
client = MlflowClient()

MODEL_NAME = "iris_classifier"
SEED = 42

# Compatibilidad entre versiones de MLflow:
# - MLflow 2.x usa artifact_path="model"; MLflow 3.x usa name="model".
# - Las versiones recientes de 3.x serializan con skops y exigen declarar
#   como confiable el tipo interno del árbol de decisión.
_log_params = inspect.signature(mlflow.sklearn.log_model).parameters
LOG_MODEL_KWARGS = {"name": "model"} if "name" in _log_params else {"artifact_path": "model"}
if "skops_trusted_types" in _log_params:
    LOG_MODEL_KWARGS["skops_trusted_types"] = ["sklearn.tree._tree.Tree"]
print(f"MLflow {mlflow.__version__}")

# --- Datos --------------------------------------------------------------------
X, y = load_iris(return_X_y=True, as_frame=True)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y
)


def train_and_log(max_depth: int) -> str:
    """Entrena un modelo en una ejecución independiente y devuelve su run_id."""
    with mlflow.start_run(run_name=f"rf_max_depth_{max_depth}") as run:
        params = {"n_estimators": 50, "max_depth": max_depth, "random_state": SEED}
        model = RandomForestClassifier(**params).fit(X_train, y_train)
        acc = accuracy_score(y_val, model.predict(X_val))

        mlflow.log_params(params)
        mlflow.log_metric("val_accuracy", acc)
        # Guardar el modelo como artefacto de la ejecución (todavía NO está registrado)
        mlflow.sklearn.log_model(
            model,
            signature=infer_signature(X_train, model.predict(X_train)),
            input_example=X_train.head(3),
            **LOG_MODEL_KWARGS,
        )
        print(f"Run {run.info.run_id[:8]}  max_depth={max_depth}  val_accuracy={acc:.4f}")
        return run.info.run_id


# --- 1. Dos ejecuciones independientes ---------------------------------------
run_1 = train_and_log(max_depth=1)
run_2 = train_and_log(max_depth=4)

# --- 2. Registrar ambos modelos bajo el mismo nombre -> versiones 1 y 2 -------
v1 = mlflow.register_model(f"runs:/{run_1}/model", MODEL_NAME).version
v2 = mlflow.register_model(f"runs:/{run_2}/model", MODEL_NAME).version
print(f"\nVersiones registradas de '{MODEL_NAME}': {v1}, {v2}")

# --- 3. Alias iniciales -------------------------------------------------------
client.set_registered_model_alias(MODEL_NAME, "champion", v1)
client.set_registered_model_alias(MODEL_NAME, "candidate", v2)


def alias_version(alias: str) -> str:
    return client.get_model_version_by_alias(MODEL_NAME, alias).version


def val_acc(version: str) -> float:
    run_id = client.get_model_version(MODEL_NAME, version).run_id
    return client.get_run(run_id).data.metrics["val_accuracy"]


# --- 4. Comparar y decidir la promoción ---------------------------------------
champ_v, cand_v = alias_version("champion"), alias_version("candidate")
champ_acc, cand_acc = val_acc(champ_v), val_acc(cand_v)
print(f"\nchampion  -> v{champ_v}  val_accuracy={champ_acc:.4f}")
print(f"candidate -> v{cand_v}  val_accuracy={cand_acc:.4f}")

if cand_acc > champ_acc:
    client.set_registered_model_alias(MODEL_NAME, "champion", cand_v)
    print(f"Decisión: la candidata (v{cand_v}) es mejor -> se promueve a champion.")
else:
    print(f"Decisión: empate o peor -> se conserva champion en v{champ_v}.")

# --- 5. Cargar por alias y predecir -------------------------------------------
URI = f"models:/{MODEL_NAME}@champion"
sample = X_val.head(5)

model = mlflow.pyfunc.load_model(URI)
print(f"\nCargado {URI} -> versión {alias_version('champion')}")
print("Predicciones:", model.predict(sample).tolist(), " reales:", y_val.head(5).tolist())

# --- 6. Rollback: reasignar champion a la otra versión, sin reentrenar --------
current = alias_version("champion")
other = v1 if current == v2 else v2
client.set_registered_model_alias(MODEL_NAME, "champion", other)
print(f"\nRollback: champion pasa de v{current} a v{other}")

# El objeto 'model' en memoria sigue siendo la versión anterior:
print("Objeto ya cargado (sin recargar):", model.predict(sample).tolist())

# Hay que volver a cargar con la MISMA URI para obtener la nueva versión:
model = mlflow.pyfunc.load_model(URI)
print(f"Recargado {URI} -> versión {alias_version('champion')}")
print("Predicciones:", model.predict(sample).tolist())


### Resultados y respuestas del punto 9

En la ejecución documentada, ambos modelos lograron accuracy de validación **0.8889** (40 de 45 observaciones). Por la regla del enunciado, el empate conserva `champion` en la versión 1.

| Momento | Versión de `champion` en el registro | Predicciones de la muestra |
|---|---:|---|
| Carga inicial | 1 | `[2, 1, 1, 2, 2]` |
| Después de asignar el alias a la otra versión, sin recargar | 2 en el registro; el objeto en memoria aún es v1 | `[2, 1, 1, 2, 2]` |
| Después de volver a cargar | 2 | `[2, 1, 1, 1, 2]` |

**¿Qué versión quedó seleccionada tras la comparación?** La versión 1, porque ambas empataron con accuracy `0.8889` y el empate conserva la versión actual.

**¿Qué cambió al reasignar el alias?** El alias `champion` pasó a apuntar a la otra versión sin reentrenar ni modificar los artefactos.

**¿Fue necesario cambiar la URI?** No. Se sigue usando `models:/iris_classifier@champion`.

**¿Por qué hay que cargar de nuevo el modelo?** `load_model` resuelve el alias y construye un objeto en memoria en ese instante. Cambiar el alias actualiza el registro, pero no transforma el objeto ya cargado. Al llamar de nuevo a `load_model` con la misma URI se obtiene la versión que ahora señala el alias.
